In [23]:
import pandas as pd
import numpy as np

In [24]:
data2025 = pd.read_csv('data/arrests2025clean.csv')
columns_to_keep3 = ["apprehension_date", "apprehension_aor", 'duplicate_likely', 'unique_identifier', "apprehension_criminality", 'birth_year','gender','citizenship_country']
data2025 = data2025[columns_to_keep3]
data2025 = data2025.rename(columns={'apprehension_aor': 'Apprehension AOR', 'apprehension_date': 'Apprehension Date', 'apprehension_criminality': 'Apprehension Criminality', 'birth_year': 'Birth Year', 'gender': 'Gender', 'citizenship_country': 'Citizenship Country'})
data2025 = data2025.replace(to_replace="1 Convicted Criminal", value="Convicted")
data2025 = data2025.replace(to_replace="2 Pending Criminal Charges", value="Pending Charges")
data2025 = data2025.replace(to_replace="3 Other Immigration Violator", value="No Criminal Charges")
data2025['Apprehension AOR'] = data2025['Apprehension AOR'].str.replace(' Area of Responsibility', '', regex=False)
data2025 = data2025.rename(columns={'Apprehension AOR': 'aor_nam'})
data2025['Apprehension MonthYear'] = pd.to_datetime(data2025['Apprehension Date']).dt.strftime('%Y-%m')
print(data2025['duplicate_likely'].value_counts()) #9455
data2025 = data2025[data2025['duplicate_likely'] == False] #9455 duplicates


duplicate_likely
False    362202
True       9455
Name: count, dtype: int64


In [25]:
arrestsbymonth2025 = data2025.groupby('Apprehension MonthYear').size().reset_index(name='arrests')
arrestsbymonth2025.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/arrestsbymonth2025.csv', index=False)

In [26]:
daily_arrests2025 = data2025.groupby('Apprehension Date').size().reset_index(name='arrests')
daily_arrests2025['week_rolling_avg'] = daily_arrests2025['arrests'].rolling(window=7).mean().round()
data2025['date_converted'] = pd.to_datetime(data2025['Apprehension Date'])
daily_arrests2025['Apprehension Date'] = pd.to_datetime(daily_arrests2025['Apprehension Date'])
daily_arrests2025.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/dailyarrests2025.csv', index=False)

In [27]:
aor_2025 = data2025.groupby(['aor_nam', 'Apprehension MonthYear']).size().reset_index(name='arrests')
aor_2025 = aor_2025[['aor_nam', 'arrests', 'Apprehension MonthYear']]
aor_2025.to_csv('/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/aor_2025_months.csv', index=False)

In [28]:
arrests_by_country2025 = data2025[
    (data2025['Apprehension Date'] >= '2025-01-20')
].groupby('Citizenship Country').agg({
    'unique_identifier': 'count'
}).reset_index()

arrests_by_country2025.columns = ['Citizenship Country', 'Arrests']

# Add percentage
arrests_by_country2025['Percentage'] = (
    arrests_by_country2025['Arrests'] / arrests_by_country2025['Arrests'].sum() * 100
).round(2)

arrests_by_country2025 = arrests_by_country2025.sort_values('Arrests', ascending=False)
arrests_by_country2025["Citizenship Country"] = arrests_by_country2025["Citizenship Country"].str.title()
arrests_by_country2025

,Citizenship Country,Arrests,Percentage
108,Mexico,79764,37.80
70,Guatemala,30176,14.30
75,Honduras,23749,11.25
179,Venezuela,14120,6.69
54,El Salvador,10172,4.82
...,...,...,...
144,Sint Maarten(Dutch),1,0.00
34,Cayman Islands,1,0.00
173,United Arab Emirates,1,0.00
113,Montserrat,1,0.00


In [29]:
topcountries2025 = arrests_by_country2025.nlargest(n=50, columns='Arrests')
topcountries2025
topcountries2025.to_csv('ICEtracker/data/topcountryarrests2025.csv', index=False)

In [30]:
data2025 = pd.read_csv('data/arrests2025clean.csv')
columns_to_keep3 = ["apprehension_date", "apprehension_aor", "apprehension_criminality", "unique_identifier"]
data2025 = data2025[columns_to_keep3]
data2025filtered = data2025[data2025['apprehension_date'] >= '2024-10-01'].copy()
data2025filtered
data2025filtered['unique_identifier'].nunique()
detdata2025 = pd.read_csv('data/detentions2025.csv')
columns_to_keep2 = [
    "stay_book_in_date_time",
    "most_serious_conviction_code",
    "msc_charge",
    "unique_identifier"
]
detdata2025 = detdata2025[columns_to_keep2]
detdatafiltered = detdata2025[detdata2025['stay_book_in_date_time'] >= '2024-10-01'].copy()
detdatafiltered['unique_identifier'].nunique()
mergeddata2025 = data2025.merge(detdata2025, on = "unique_identifier", how = 'right')

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_70034/839313104.py:7: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  detdata2025 = pd.read_csv('data/detentions2025.csv')


In [31]:
print(mergeddata2025['apprehension_aor'].isna().sum())
print(mergeddata2025['stay_book_in_date_time'].isna().sum())

338227
0


In [32]:
mergeddata2025 = mergeddata2025[mergeddata2025['apprehension_date'] >= '2024-10-01']
mergeddata2025['apprehension_date'] = pd.to_datetime(mergeddata2025['apprehension_date'])
mergeddata2025 = mergeddata2025.replace(to_replace="1 Convicted Criminal", value="Convicted")
mergeddata2025 = mergeddata2025.replace(to_replace="2 Pending Criminal Charges", value="Pending Charges")
mergeddata2025 = mergeddata2025.replace(to_replace="3 Other Immigration Violator", value="No Criminal Charges")
print(mergeddata2025['apprehension_aor'].isna().sum())
mergeddata2025.drop_duplicates(subset=['unique_identifier', 'apprehension_date', 'apprehension_aor'], inplace=True)
mergeddata2025 = mergeddata2025.dropna(subset=['apprehension_aor'])
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_date': 'Apprehension Date'})
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_aor': 'Apprehension AOR'})
mergeddata2025 = mergeddata2025.rename(columns={'apprehension_criminality': 'Criminality'})
mergeddata2025['Apprehension AOR'] = mergeddata2025['Apprehension AOR'].str.replace(' Area of Responsibility', '', regex=False)

819


In [33]:
crimelist = pd.read_csv("data/crimeclass.csv")
df = crimelist.rename(columns={'Type of Offense Code  V=violent  D=drug-related  Blank = nonviolent or not drug related': 'Type'})
violentcrimes = df[df['Type'].str.contains('V', na=False)]

In [34]:
mergeddata2025= mergeddata2025.merge(violentcrimes, left_on = "most_serious_conviction_code", right_on = "NCIC Offense Code", how = 'left')
mergeddata2025
mergeddata2025['Type'] = np.where(((mergeddata2025['Type'] != "V") & (mergeddata2025['Criminality'] == "Convicted")), "Convicted - Non-violent", mergeddata2025['Type'])
mergeddata2025['Type'] = np.where((mergeddata2025['Type'] == "V"), "Convicted - Violent crime", mergeddata2025['Type'])
mergeddata2025['Type'] = np.where(((mergeddata2025['Type'] != "V") & (mergeddata2025['Criminality'] == "Pending Charges")), "Pending Charges", mergeddata2025['Type'])
mergeddata2025['Type'] = mergeddata2025['Type'].fillna('No Criminal Charges')

In [35]:
mergeddata2025 = mergeddata2025.drop(columns=['most_serious_conviction_code', 'NCIC Offense Code', 'Description of Crime', 'unique_identifier', 'stay_book_in_date_time', 'msc_charge'])

In [36]:
# Step 0: Convert to datetime first!
mergeddata2025['Apprehension Date'] = pd.to_datetime(mergeddata2025['Apprehension Date'])

# Step 1: Create date ranges excluding the gap period
date_range_1 = pd.date_range(
    start=mergeddata2025['Apprehension Date'].min(), 
    end='2017-10-15', 
    freq='D'
)

date_range_2 = pd.date_range(
    start='2024-10-01',
    end=mergeddata2025['Apprehension Date'].max(), 
    freq='D'
)

# Step 2: Create combinations separately for each administration period
from itertools import product

# Trump 2 combinations (from Oct 1, 2024 onwards)
trump2_combinations = pd.DataFrame(
    list(product(
        date_range_2,
        mergeddata2025['Apprehension AOR'].unique(),
        mergeddata2025['Type'].unique()
    )),
    columns=['Apprehension Date', 'Apprehension AOR','Type']
)

# Combine both administration periods
all_combinations = trump2_combinations

# Step 3: Group individual AOR data
chartdf = mergeddata2025.groupby(['Apprehension Date', 'Apprehension AOR', 'Type']).size().reset_index(name='Arrests')

# Step 4: Merge with complete combinations and fill missing with 0
chartdf = all_combinations.merge(chartdf, how='left', on=['Apprehension Date', 'Apprehension AOR','Type'])
chartdf['Arrests'] = chartdf['Arrests'].fillna(0)

# Step 5: Create national aggregate with filled dates
trump1_national = pd.DataFrame(
    list(product(
        date_range_1,
        ['Trump 1'],
        mergeddata2025['Type'].unique()
    )),
    columns=['Apprehension Date', 'Administration', 'Type']
)

trump2_national = pd.DataFrame(
    list(product(
        date_range_2,
        ['Trump 2'],
        mergeddata2025['Type'].unique()
    )),
    columns=['Apprehension Date', 'Administration', 'Type']
)

national_combinations = pd.concat([trump1_national, trump2_national], ignore_index=True)
national_aggregate = mergeddata2025.groupby(['Apprehension Date', 'Type']).size().reset_index(name='Arrests')
national_aggregate['Administration'] = 'Trump 2'
national_aggregate = mergeddata2025.groupby(['Apprehension Date', 'Type']).size().reset_index(name='Arrests')

# Merge national aggregate with complete dates
national_aggregate = national_combinations.merge(national_aggregate, how='left', on=['Apprehension Date', 'Type'])
national_aggregate['Arrests'] = national_aggregate['Arrests'].fillna(0)
national_aggregate['Apprehension AOR'] = 'National'

# Step 6: Combine individual AORs with national aggregate
chartdf = pd.concat([chartdf, national_aggregate], ignore_index=True)

# Step 7: Sort properly
chartdf = chartdf.sort_values(by=['Apprehension AOR', 'Type', 'Apprehension Date']).reset_index(drop=True)

# Step 8: NOW calculate rolling average
chartdf['Week Rolling Average'] = chartdf.groupby(['Apprehension AOR', 'Type'])['Arrests'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
).round(2)

/var/folders/1l/yh12s4qx29z26j7mg26bfby80000gn/T/ipykernel_70034/2268239461.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  national_combinations = pd.concat([trump1_national, trump2_national], ignore_index=True)


In [37]:
chartdf = chartdf.drop(columns=['Administration'])


In [38]:
chartdf.to_csv("/Users/hannahliu/Desktop/PP434/website/ICEtracker/data/areachart2025.csv", index=False)